## <center> Gentle Introduction to RL in Large Language Models </center>

RL is playing an increasingly important role in training large language models (LLMs), particularly as a way to align LLMs with human preferences (e.g., via reinforcement learning with human feedback, RLHF). In this notebook, we will start from the simple policy gradient approach (previously discussed as a solution to the bandit problem) and generalize it to the **Proximal Policy Optimization** (PPO), which is one of the main algorithms powering RL-based alignment in LLM systems.

## A More Formal Dicussion of the Policy Gradient approach

As a first step, we will restate the policy gradient approach discussed before in a more formal language. 

Let $\theta = (H_1, \ldots, H_k)$ denote the vector of preference scores associated with all actions, and $\pi_{\theta} = \mathrm{softmax}(\theta)$ denote the policy. Furthermore, denote

$$
J(\theta) = \mathbb{E}_{a \sim \pi_{\theta}}(R(a))
$$ 
as the expected reward, where action $a$ is sampled according to policy $\pi_{\theta}$. Then, by first principle, the objective of the RL task is to maximize $J(\theta)$.

The standard way of doing so is to do gradient ascend (because we are maximizing the objective) by moving in the direction of $\nabla_{\theta} J(\theta) = \nabla_{\theta} \mathbb{E}_{a \sim \pi_{\theta}}(R(a))$.

It is not straightfoward to take derivative of an expectation. In RL algorithms, the following "log-derivative trick" allows you to convert the derivative of an expectation into an expectation of a derivation (which can then be estimated empirically):

\begin{align*}
\nabla_{\theta} \mathbb{E}_{a \sim \pi_{\theta}}(R(a)) &= \nabla_{\theta} \sum_{a=1}^k \pi_{\theta}(a) q(a) \\
&= \sum_{a=1}^k \nabla_{\theta} \pi_{\theta}(a) q(a) \\
&= \sum_{a=1}^k \pi_{\theta}(a) \nabla_{\theta} \log \pi_{\theta}(a) q(a) \\
&= \mathbb{E}_{a \sim \pi_{\theta}} \left( \nabla_{\theta} \log \pi_{\theta}(a) q(a) \right)
\end{align*}

To estimate the last quantity, we sample actions from the current policy (at step $t$) and replace the unknown action value $q(a)$ with the **advantage value** $R_{t+1} - \overline{R_{t+1}}$. As for $\nabla_{\theta} \log \pi_{\theta}(a)$, we can compute it explicitly. Recall that $\pi_{\theta}(a) = \frac{e^{H_t(a)}}{\sum_b e^{H_t(b)}}$, so we have $\log \pi_{\theta}(a) = H_t(a) - \log \left( \sum_b e^{H_t(b)} \right)$. On the action that's actually taken at step $t$, i.e., for $a = A_t$, we have $\nabla_{\theta} \log \pi_{\theta}(a) = 1 - \frac{H_t(a)}{\sum_b e^{H_t(b)}} = 1 - \pi_{\theta}(a)$. For all other actions, i.e., $a \neq A_t$, we will have $\nabla_{\theta} \log \pi_{\theta}(a) = 0 - \frac{H_t(a)}{\sum_b e^{H_t(b)}} = - \pi_{\theta}(a)$. Putting these together, we will get the policy gradient algorithm we have been using.

## Proximal Policy Optimization (PPO)

The vanilla policy gradient approach essentially works by performing gradient ascend based on (sample estimate of)
$$
\mathbb{E}_{a \sim \pi_{\theta}} \left( \nabla_{\theta} \log \pi_{\theta}(a) q(a) \right)
$$

With state variables, the gradient takes the form
$$
\mathbb{E}_{\tau \sim \pi_{\theta}} \left( \nabla_{\theta} \log \pi_{\theta}(a|s) q(a|s) \right)
$$
where $\tau$ represents a trajectory of actions (where $a$ is the action in step $t$). However, empirically, it can also lead to unstable and very large policy updates (see the [PPO paper](https://arxiv.org/pdf/1707.06347) for empirical evidence). Instead, PPO consider the following constrained objective:

$$
J(\theta) = \mathbb{E} \left(\frac{\pi_{\theta}(a|s)}{\pi_{\theta_{old}}(a|s)} q(a|s) \right)
$$
subject to the constraint that $\pi_{\theta_{old}}$ and $\pi_{\theta}(a|s)$ are not too different (e.g., by imposing an upper bound on their KL-divergence).

Furthermore, let $r(a|s) = \frac{\pi_{\theta}(a|s)}{\pi_{\theta_{old}}(a|s)}$, PPO performs clipping on this ratio in order to prevent overly large policy updates (that may arise from reward hacking). So the actual objective that PPO maximizes is

$$
J(\theta) = \mathbb{E} \left[\min(r(a|s)q(a|s), \mathrm{clip}(r(a|s), 1 - \epsilon, 1 + \epsilon) q(a|s)) \right]
$$
subject to the KL constraint.

When PPO is applied to an LLM, $s$ represents a prompt, $a$ represents the next token to be generated, and $\pi$ is the LLM's token generation function. The reward model is usually another LLM that has been fine-tuned based on tuples of (prompt, good response, bad response).